# Auditoría de capacidad y materialización incremental A2D2 + UrbanSyn

Estima 50 muestras por dataset/split sin descargar; estima y materializa 10 por dataset/split (40 muestras totales, train/val); recorre las 40 muestras para medir decodificación, tamaño y RSS. **No accede a test, no hace forward/backward ni entrena.** Límite de descarga: 2 GiB. Es una prueba de capacidad, no una selección balanceada ni una evaluación de calidad.

Ejecuta en orden. Requiere GITHUB_TOKEN con Contents: Read-only y Actions: Read-only.


## 1. Montar Drive

Los informes se guardarán en `MyDrive/ProcesamientoImagenes/auditoria_capacidad`.


In [ ]:
from google.colab import drivefrom pathlib import Pathimport os, sys, json, zipfile, hashlib, subprocess, shutil, datetime, requestsdrive.mount('/content/drive')OUT_DRIVE = Path('/content/drive/MyDrive/ProcesamientoImagenes/auditoria_capacidad')OUT_DRIVE.mkdir(parents=True, exist_ok=True)print('Destino de informes:', OUT_DRIVE)

## 2. Actualizar repositorio privado

Se requiere el PAT de solo lectura en Contents y Actions. La rama debe ser `urban-models-and-benchmark`.


In [ ]:
from getpass import getpass
REPO='Leohf04/Procesamiento-de-Im-genes-'
BRANCH='urban-models-and-benchmark'
WORKDIR=Path('/content/Procesamiento-de-Im-genes-')
try:
    from google.colab import userdata
    TOKEN=userdata.get('GITHUB_TOKEN')
except Exception:
    TOKEN=None
if not TOKEN:
    TOKEN=getpass('GitHub PAT (Contents: Read-only + Actions: Read-only): ')
if not TOKEN:
    raise RuntimeError('Falta el token GitHub.')
askpass=Path('/tmp/colab_git_askpass.sh')
askpass.write_text('#!/bin/sh\ncase "$1" in\n *Username*) echo x-access-token ;;\n *Password*) echo "$COLAB_GITHUB_TOKEN" ;;\n *) exit 1 ;;\nesac\n')
askpass.chmod(0o700)
env=os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','COLAB_GITHUB_TOKEN':TOKEN})
if WORKDIR.exists():
    subprocess.run(['git','-C',str(WORKDIR),'fetch','origin',BRANCH],check=True,env=env)
    subprocess.run(['git','-C',str(WORKDIR),'checkout',BRANCH],check=True)
    subprocess.run(['git','-C',str(WORKDIR),'pull','--ff-only','origin',BRANCH],check=True,env=env)
else:
    subprocess.run(['git','clone','--single-branch','--branch',BRANCH,f'https://github.com/{REPO}.git',str(WORKDIR)],check=True,env=env)
del TOKEN,env
os.chdir(WORKDIR)
HEAD=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
print('HEAD:',HEAD)


## 3. Recuperar y verificar el artifact existente

El ZIP contiene solo referencias remotas y estadísticas; no contiene RGB ni máscaras. Se verifica el SHA-256 publicado por GitHub antes de extraerlo.


In [ ]:
ARTIFACT_ID=14770171764
RUN_ID=45038801547
EXPECTED_SHA256='b6cbcb54dfd0598e6a9011667073028be1804fc1c112d82e516a42bdbf2ba4e5'
API=f'https://api.github.com/repos/{REPO}/actions/artifacts/{ARTIFACT_ID}'
try:
    from google.colab import userdata
    api_token=userdata.get('GITHUB_TOKEN')
except Exception:
    api_token=None
if not api_token:
    api_token=getpass('GitHub PAT (Fine-grained, Actions: Read-only, acceso a este repo): ')
api_token=api_token.strip()
if not api_token:
    raise RuntimeError('No se recibió ningún token.')
headers={'Authorization':f'Bearer {api_token}','Accept':'application/vnd.github+json','X-GitHub-Api-Version':'2022-11-28'}
# Diagnosticar por separado autenticación, acceso al repositorio y permiso de Actions.
who=requests.get('https://api.github.com/user',headers=headers,timeout=30)
if who.status_code != 200:
    raise RuntimeError(f'GitHub rechazó la autenticación ({who.status_code}). Comprueba que el PAT no esté revocado/expirado y que sea un token válido. Respuesta: {who.text[:500]}')
repo_check=requests.get(f'https://api.github.com/repos/{REPO}',headers=headers,timeout=30)
if repo_check.status_code != 200:
    raise RuntimeError(f'El token autentica, pero no puede acceder al repositorio privado {REPO} ({repo_check.status_code}). En la configuración del PAT selecciona exclusivamente este repositorio y confirma acceso. Respuesta: {repo_check.text[:500]}')
meta=requests.get(API,headers=headers,timeout=60)
if meta.status_code != 200:
    body={}
    try: body=meta.json()
    except Exception: pass
    accepted=meta.headers.get('X-Accepted-GitHub-Permissions','no indicado')
    if meta.status_code == 403:
        raise RuntimeError('GitHub autentica el PAT y permite ver el repositorio, pero deniega leer Actions/artifacts (403). Solución: edita o crea un Fine-grained PAT: Resource owner = propietario del repositorio; Repository access = Only select repositories y selecciona Procesamiento-de-Im-genes-; Repository permissions > Actions = Read-only. Contents = Read-only puede mantenerse para clonar el repo. Guarda el PAT actualizado en Colab Secrets con el nombre GITHUB_TOKEN, reemplazando el anterior, y vuelve a ejecutar esta celda. Encabezado X-Accepted-GitHub-Permissions: '+accepted+'. Respuesta GitHub: '+str(body)[:700])
    if meta.status_code == 404:
        raise RuntimeError('GitHub no encuentra el artifact o el token no tiene permiso para verlo. Confirma que el PAT tiene Actions: Read-only y que el artifact 14770171764 sigue vigente. Respuesta: '+str(body)[:700])
    raise RuntimeError(f'No se pudo consultar el artifact ({meta.status_code}). X-Accepted-GitHub-Permissions={accepted}. Respuesta: {str(body)[:700]}')
meta_json=meta.json()
if meta_json.get('expired'):
    raise RuntimeError('El artifact de indexación ha expirado; no es posible descargarlo desde GitHub.')
if meta_json.get('workflow_run',{}).get('id') != RUN_ID:
    raise RuntimeError('El artifact no pertenece al run esperado 45038801547.')
r=requests.get(API+'/zip',headers=headers,timeout=300)
if r.status_code != 200:
    raise RuntimeError(f'GitHub encontró el artifact, pero falló la descarga ZIP ({r.status_code}): {r.text[:700]}')
zip_path=Path('/content/urban_annotation_artifact_14770171764.zip')
zip_path.write_bytes(r.content)
actual=hashlib.sha256(zip_path.read_bytes()).hexdigest()
print('Artifact:',meta_json.get('name'),'bytes:',zip_path.stat().st_size)
print('SHA-256:',actual)
if actual != EXPECTED_SHA256:
    raise RuntimeError('El SHA-256 no coincide con el digest verificado del artifact. No se extraerá.')
extract=Path('/content/urban_artifact_14770171764')
if extract.exists(): shutil.rmtree(extract)
extract.mkdir()
with zipfile.ZipFile(zip_path) as zf:
    for info in zf.infolist():
        target=(extract/info.filename).resolve()
        if extract.resolve() not in target.parents and target != extract.resolve():
            raise RuntimeError('Ruta insegura detectada en el ZIP.')
    zf.extractall(extract)
print('Artifact verificado y extraído:',extract)
del api_token,headers


## 4. Reconstruir los seis manifiestos y verificar los índices

El artifact original se generó sin cap (`per_class_cap=null`), por lo que los JSONL de `balanced/` conservan las 1.000/751/783 filas completas. Se usan los archivos semánticos como manifiesto base y se comprueba que las variantes de instancias tienen los mismos IDs y metadatos. Las estadísticas se contrastan fila por fila con los seis índices per-image.


In [ ]:
ART=extract
MANIFEST_DIR=Path('/content/cap200_input/manifests')
INDEX_DIR=ART/'annotation_index'
MANIFEST_DIR.mkdir(parents=True,exist_ok=True)
validation=[]
def read_jsonl(path):
    with path.open(encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]
def write_jsonl(path,rows):
    path.parent.mkdir(parents=True,exist_ok=True)
    with path.open('w',encoding='utf-8') as f:
        for row in rows: f.write(json.dumps(row,sort_keys=True,ensure_ascii=False)+'\n')
old_plan=json.loads((ART/'balanced/balanced_training_plan.json').read_text())
if old_plan.get('per_class_cap') is not None:
    raise RuntimeError('El plan de origen no está sin cap; no se puede reconstruir el universo completo.')
for ds in ('a2d2','urbansyn'):
  for split in ('train','val','test'):
    sem=read_jsonl(ART/f'balanced/{ds}_semantic_{split}.jsonl')
    ins=read_jsonl(ART/f'balanced/{ds}_instances_{split}.jsonl')
    ann=read_jsonl(INDEX_DIR/f'{ds}_{split}.annotations.jsonl')
    sem_map={str(x['sample_id']):x for x in sem}
    ins_map={str(x['sample_id']):x for x in ins}
    ann_map={str(x['sample_id']):x for x in ann}
    if len(sem_map)!=len(sem) or len(ins_map)!=len(ins) or len(ann_map)!=len(ann):
        raise RuntimeError(f'{ds}/{split}: IDs duplicados')
    if set(sem_map)!=set(ins_map) or set(sem_map)!=set(ann_map):
        raise RuntimeError(f'{ds}/{split}: IDs de manifiestos/índice no coinciden')
    for sid,row in sem_map.items():
        if any(row.get(k)!=ann_map[sid].get(k) for k in ('semantic_pixels','instance_counts')):
            raise RuntimeError(f'{ds}/{split}: estadísticas no coinciden para {sid}')
        if any(ins_map[sid].get(k)!=row.get(k) for k in ('semantic_pixels','instance_counts')):
            raise RuntimeError(f'{ds}/{split}: estadísticas semántica/instancias distintas para {sid}')
        base_sem={k:v for k,v in row.items() if k not in ('semantic_pixels','instance_counts')}
        base_ins={k:v for k,v in ins_map[sid].items() if k not in ('semantic_pixels','instance_counts')}
        if base_sem!=base_ins:
            raise RuntimeError(f'{ds}/{split}: metadatos de manifiesto distintos para {sid}')
    base=[{k:v for k,v in row.items() if k not in ('semantic_pixels','instance_counts')} for row in sem]
    # Restore the exact manifest order used by prepare_reproducible_subsets.py.
    base.sort(key=lambda row: hashlib.sha256(f"2026:{row['sample_id']}".encode('utf-8')).hexdigest())
    write_jsonl(MANIFEST_DIR/f'{ds}_{split}.jsonl',base)
    validation.append({'dataset':ds,'split':split,'manifest_rows':len(base),'index_rows':len(ann),'unique_ids':len(sem_map),'ids_match':True,'stats_match':True,'task_manifests_match':True})
if len(validation)!=6 or any(x['manifest_rows']!=x['index_rows'] for x in validation):
    raise RuntimeError('Validación de reconstrucción incompleta.')
validation_path=Path('/content/cap200_input/artifact_reconstruction_validation.json')
validation_path.parent.mkdir(parents=True,exist_ok=True)
validation_path.write_text(json.dumps({'artifact_id':ARTIFACT_ID,'source_run':RUN_ID,'checks':validation},indent=2,ensure_ascii=False)+'\n')
print(json.dumps(validation,indent=2,ensure_ascii=False))


## 5. Registrar hardware y almacenamientoCaptura entorno, CPU/GPU, RAM y espacio libre antes de materializar.

In [ ]:
import platform, shutil, timeimport torchdisk0=shutil.disk_usage('/content')try: import psutil mem=psutil.virtual_memory() ram={'ram_total_bytes':int(mem.total),'ram_available_bytes':int(mem.available)}except Exception: ram={}gpu=[]if torch.cuda.is_available(): for i in range(torch.cuda.device_count()):  p=torch.cuda.get_device_properties(i)  gpu.append({'index':i,'name':p.name,'total_memory_bytes':int(p.total_memory)})environment={'timestamp_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(), 'git_head':subprocess.check_output(['git','rev-parse','HEAD'],cwd=WORKDIR,text=True).strip(), 'python':platform.python_version(),'platform':platform.platform(),'torch':torch.__version__, 'cuda_available':torch.cuda.is_available(),'gpu':gpu,'cpu_count':os.cpu_count(), 'disk_total_bytes':disk0.total,'disk_free_bytes':disk0.free,**ram, 'training_started':False,'test_split_touched':False}(OUT_DRIVE/'capacity_environment.json').write_text(json.dumps(environment,indent=2)+'\n')print(json.dumps(environment,indent=2))

## 6. Estimar hasta 50 muestras por dataset/split (sin descargar)Hasta 200 muestras totales. Se seleccionan determinísticamente muestras con instancias positivas. Si la estimación supera 2 GiB, la celda aborta sin descargar.

In [ ]:
from importlib.util import find_specmissing=[p for p in ('boto3','huggingface_hub') if find_spec(p) is None]if missing: subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)ESTIMATE50=Path('/content/urban_capacity_estimate_50')if ESTIMATE50.exists(): shutil.rmtree(ESTIMATE50)cmd50=[sys.executable,'scripts/materialize_urban_experiment_data.py','--manifest-dir',str(MANIFEST_DIR), '--annotation-index-dir',str(INDEX_DIR),'--output-dir',str(ESTIMATE50),'--dry-run', '--max-samples-per-split','50','--require-instances','--max-total-bytes',str(2*1024**3)]run50=subprocess.run(cmd50,cwd=WORKDIR,text=True,capture_output=True)print(run50.stdout[-12000:])if run50.returncode: print(run50.stderr[-8000:]); raise RuntimeError('Estimación 50 excede presupuesto o falló; no se descargó.')estimate50=json.loads((ESTIMATE50/'materialization_preflight.json').read_text())if estimate50.get('status')!='PREFLIGHT_PASS' or estimate50.get('test_touched') is not False: raise RuntimeError('Estimación 50 no confirma preflight seguro.')shutil.copy2(ESTIMATE50/'materialization_preflight.json',OUT_DRIVE/'capacity_estimate_50.json')print('Hasta 200 muestras:',estimate50['estimated_remote_asset_gib'],'GiB; activos:',estimate50['asset_count'])

## 7. Estimar exactamente el lote incremental de 10 por splitSe consulta el tamaño de los 40 ejemplos que se materializarán. Se exige espacio libre para el doble de lo estimado más 1 GiB.

In [ ]:
ESTIMATE10=Path('/content/urban_capacity_estimate_10')if ESTIMATE10.exists(): shutil.rmtree(ESTIMATE10)cmd10=[sys.executable,'scripts/materialize_urban_experiment_data.py','--manifest-dir',str(MANIFEST_DIR), '--annotation-index-dir',str(INDEX_DIR),'--output-dir',str(ESTIMATE10),'--dry-run', '--max-samples-per-split','10','--require-instances','--max-total-bytes',str(2*1024**3)]run10=subprocess.run(cmd10,cwd=WORKDIR,text=True,capture_output=True)print(run10.stdout[-10000:])if run10.returncode: print(run10.stderr[-8000:]); raise RuntimeError('Falló estimación de 10; no se descargó.')estimate10=json.loads((ESTIMATE10/'materialization_preflight.json').read_text())required_free=int(estimate10['estimated_remote_asset_bytes']*2+1024**3)free_now=shutil.disk_usage('/content').freeif free_now<required_free: raise RuntimeError(f'Espacio insuficiente: libre={free_now}, requerido={required_free}; no descargar.')shutil.copy2(ESTIMATE10/'materialization_preflight.json',OUT_DRIVE/'capacity_estimate_10.json')print('40 muestras:',estimate10['estimated_remote_asset_gib'],'GiB; libre:',round(free_now/1024**3,2),'GiB; requerido:',round(required_free/1024**3,2),'GiB')

## 8. Materializar 10 muestras por dataset/split10 train + 10 val de A2D2 y 10 train + 10 val de UrbanSyn. Solo train/val, con instancias positivas. Datos temporales de Colab, no Drive/Git.

In [ ]:
DATA_DIR=Path('/content/urban_data_capacity')if DATA_DIR.exists(): shutil.rmtree(DATA_DIR)disk_before_materialization=shutil.disk_usage('/content')cmd=[sys.executable,'scripts/materialize_urban_experiment_data.py','--manifest-dir',str(MANIFEST_DIR), '--annotation-index-dir',str(INDEX_DIR),'--output-dir',str(DATA_DIR), '--max-samples-per-split','10','--require-instances','--max-total-bytes',str(2*1024**3)]mat=subprocess.run(cmd,cwd=WORKDIR,text=True,capture_output=True)print(mat.stdout[-12000:])if mat.returncode: print(mat.stderr[-8000:]); raise RuntimeError('Falló materialización incremental.')materialized=json.loads((DATA_DIR/'materialization_report.json').read_text())if materialized.get('status')!='PASS' or materialized.get('test_touched') is not False or materialized.get('training_started') is not False: raise RuntimeError('Materialización no confirma estado seguro.')for ds in ('a2d2','urbansyn'): for split in ('train','val'):  local=read_jsonl(DATA_DIR/'local_manifests'/f'{ds}_{split}.jsonl')  ann=read_jsonl(DATA_DIR/'local_annotation_index'/f'{ds}_{split}.annotations.jsonl')  if len(local)!=10 or len(ann)!=10 or [x['sample_id'] for x in local]!=[x['sample_id'] for x in ann]:   raise RuntimeError(f'{ds}/{split}: se esperaban 10 IDs alineados.')shutil.copy2(DATA_DIR/'materialization_report.json',OUT_DRIVE/'capacity_materialization_report.json')disk_after_materialization=shutil.disk_usage('/content')materialized_bytes=sum(p.stat().st_size for p in DATA_DIR.rglob('*') if p.is_file())print('Materialización PASS; bytes locales:',materialized_bytes)print('Espacio libre antes/después:',disk_before_materialization.free,disk_after_materialization.free)

## 9. Auditar decodificación completa y recursosRecorre cada muestra materializada una vez por ruta semántica e instancia, en train y val. Comprueba tensores, máscaras, cajas y conteos; mide tiempo, bytes y RSS. No ejecuta inferencia, backward, optimizer.step ni entrenamiento.

In [ ]:
capacity_report=OUT_DRIVE/'urban_capacity_audit.json'capacity_stdout=OUT_DRIVE/'urban_capacity_audit.stdout.log'capacity_stderr=OUT_DRIVE/'urban_capacity_audit.stderr.log'cmd=[sys.executable,'-u','scripts/audit_urban_materialized_capacity.py','--data-dir',str(DATA_DIR), '--annotation-index-dir',str(DATA_DIR/'local_annotation_index'),'--output',str(capacity_report),'--size','256']audit=subprocess.run(cmd,cwd=WORKDIR,text=True,capture_output=True)capacity_stdout.write_text(audit.stdout or '',encoding='utf-8')capacity_stderr.write_text(audit.stderr or '',encoding='utf-8')print(audit.stdout[-18000:])if audit.returncode: print('ERROR:',audit.stderr[-12000:]); raise RuntimeError('Falló auditoría; revisar logs.')cap=json.loads(capacity_report.read_text())if cap.get('status')!='PASS' or cap.get('training_started') is not False or cap.get('test_split_touched') is not False: raise RuntimeError('Informe no confirma PASS seguro.')print('PASS:',capacity_report,'\nLogs:',capacity_stdout,capacity_stderr)

## 10. Cierre y decisiónComparte `capacity_environment.json`, `capacity_estimate_50.json`, `capacity_estimate_10.json`, `capacity_materialization_report.json` y `urban_capacity_audit.json`; añade logs si hubo errores. Con estos datos dimensionaremos el siguiente lote. No se escala automáticamente al dataset completo.Si falla presupuesto, espacio, descarga o decodificación, detener. Si pasa, decidir el siguiente tamaño según bytes/muestra, espacio persistente, RSS y velocidad. Aún falta medir VRAM/batch si hay GPU. No entrenar ni tocar test.